In [1]:
import os
import pdfplumber
import json
import pandas as pd

In [3]:
# Paths setup
raw_path = "01_Raw_PDFs/"
output_folder = "03_Final_Dataset/"

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

all_data = []

In [5]:
files = [f for f in os.listdir(raw_path) if f.endswith('.pdf')]

In [7]:
if len(files) == 0:
    print("❌ Abhi bhi files nahi milin! Kya aapne '01_Raw_PDFs' mein PDFs upload kar di hain?")
else:
    print(f"✅ {len(files)} files mil gayi hain. Extraction shuru ho rahi hai...\n")
    
    for filename in files:
        print(f"Processing: {filename}...")
        try:
            with pdfplumber.open(os.path.join(raw_path, filename)) as pdf:
                text = ""
                for page in pdf.pages:
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n"
                
                all_data.append({
                    "law_name": filename.replace(".pdf", ""),
                    "content": text.strip()
                })
        except Exception as e:
            print(f"Error in {filename}: {e}")

    # JSON aur CSV Save karein
    if all_data:
        # Save JSON
        json_path = os.path.join(output_folder, "legal_data.json")
        with open(json_path, "w", encoding="utf-8") as j:
            json.dump(all_data, j, indent=4)
        
        # Save CSV
        csv_path = os.path.join(output_folder, "legal_data.csv")
        df = pd.DataFrame(all_data)
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")
        
        print(f"\n✨ Mubarak ho! JSON aur CSV dono '{output_folder}' mein ban gayi hain.")
    else:
        print("Kuch masla hua, data extract nahi ho saka.")

✅ 21 files mil gayi hain. Extraction shuru ho rahi hai...

Processing: THE ANAND MARRIAGE ACT, 1909.pdf...
Processing: THE ARYA MARRIAGE VALIDATION ACT, 1937.pdf...
Processing: THE CHILD MARRIAGE RESTRAINT ACT, 1929.pdf...
Processing: THE CHRISTIAN MARRIAGE ACT, 1872.pdf...
Processing: THE CLAIMS FOR MAINTENANCE (RECOVERY ABROAD).pdf...
Processing: THE DISSOLUTION OF MUSLIM MARRIAGES ACT, 1939.pdf...
Processing: THE DIVORCE ACT.pdf...
Processing: THE DOWRY AND BRIDAL GIFTS (RESTRICTION) ACT, 1976.pdf...
Processing: THE GUARDIANS AND WARDS ACT, 1890.pdf...
Processing: THE HINDU DISPOSITION OF PROPERTY ACT, 1916.pdf...
Processing: THE HINDU INHERITANCE (REMOVAL OF DISABILITIES).pdf...
Processing: THE HINDU MARRIAGE DISABILITIES REMOVAL ACT,1946.pdf...
Processing: THE HINDU MARRIED WOMEN’S RIGHT TO SEPARATE.pdf...
Processing: THE HINDU WIDOWS’ RE-MARRIAGE ACT, 1856.pdf...
Processing: THE HINDU WOMEN’S RIGHTS TO PROPERTY ACT, 1937.pdf...
Processing: THE MARRIAGE FUNCTIONS .pdf...
Processin

In [9]:
import pandas as pd
from pymongo import MongoClient

In [11]:
# 1. CSV read karein
csv_path = "03_Final_Dataset/legal_data.csv"
df = pd.read_csv(csv_path)

In [13]:
try:
    client = MongoClient("mongodb://localhost:27017/")
    db = client['LegalChatbotDB']
    collection = db['Laws_Collection']

    # DataFrame ko dictionary format mein badlein
    data_dict = df.to_dict("records")

    # Purana data delete karein taake duplicates na hon aur naya insert karein
    collection.delete_many({}) 
    collection.insert_many(data_dict)

    print(f"🚀 Success! {len(data_dict)} laws MongoDB Compass mein upload ho gayi hain.")
except Exception as e:
    print(f"❌ Error: {e}")

🚀 Success! 21 laws MongoDB Compass mein upload ho gayi hain.
